# Exploring Reasoning Gym


[Reasoning Gym](https://github.com/open-thought/reasoning-gym) · [Task gallery](https://github.com/open-thought/reasoning-gym/blob/main/GALLERY.md)

In [1]:
from collections import defaultdict
from dataclasses import asdict
from importlib.metadata import version
from pprint import pprint

from IPython.display import Markdown, display
import reasoning_gym
from reasoning_gym.factory import DATASETS

print(f"Reasoning Gym {version('reasoning-gym')}")

Reasoning Gym 0.1.25


## Tasks by category

This list comes from the installed library's registry. Copy a task name into `task_name` below.

In [2]:
tasks_by_category = defaultdict(list)
for name, (dataset_class, config_class) in sorted(DATASETS.items()):
    if name != "composite":
        category = dataset_class.__module__.split(".")[1]
        tasks_by_category[category].append(name)

print(f"{sum(map(len, tasks_by_category.values()))} tasks across {len(tasks_by_category)} categories")
for category, names in sorted(tasks_by_category.items()):
    display(Markdown(f"### {category} ({len(names)})"))
    display(Markdown(", ".join(f"`{name}`" for name in names)))

105 tasks across 12 categories


### algebra (6)

`complex_arithmetic`, `intermediate_integration`, `polynomial_equations`, `polynomial_multiplication`, `simple_equations`, `simple_integration`

### algorithmic (34)

`ab`, `base_conversion`, `binary_alternation`, `binary_matrix`, `caesar_cipher`, `count_primes`, `cryptarithm`, `game_of_life`, `game_of_life_halting`, `graph_color`, `group_anagrams`, `isomorphic_strings`, `jugs`, `letter_counting`, `letter_jumble`, `manipulate_matrix`, `number_filtering`, `number_sorting`, `palindrome_generation`, `palindrome_partitioning`, `pool_matrix`, `ransom_note`, `rotate_matrix`, `rotten_oranges`, `sentence_reordering`, `spell_backward`, `spiral_matrix`, `string_insertion`, `string_manipulation`, `string_splitting`, `string_synthesis`, `word_ladder`, `word_sequence_reversal`, `word_sorting`

...

`acre`, `list_functions`

### logic (7)

`aiw`, `circuit_logic`, `knights_knaves`, `propositional_logic`, `self_reference`, `syllogism`, `zebra_puzzles`

### probability (1)

`coin_flip`

## Choose a task

In [3]:
task_name = "countdown"

dataset_class, config_class = DATASETS[task_name]
print(dataset_class.__doc__)
pprint(asdict(config_class()), sort_dicts=False)

Generates Countdown Number Game tasks
{'min_numbers': 4,
 'max_numbers': 6,
 'min_value': 1,
 'max_value': 100,
 'min_target': 100,
 'max_target': 999,
 'operators': ('+', '-', '*', '/'),
 'shuffle': True,
 'seed': None,
 'size': 500}


## Generate examples

Set the number of examples and seed. Leave `task_config` empty for the task defaults, or override fields shown above. For leg counting, try `{"min_animals": 2, "max_animals": 4}`.

In [4]:
seed = 42
num_examples = 3
task_config = {}

dataset = reasoning_gym.create_dataset(
    task_name, seed=seed, size=num_examples, **task_config
)

Each entry contains a **question**, a **reference answer**, and task-specific **metadata**.
Only the question would be given to a model; answers and metadata can contain the solution.

In [5]:
for index, entry in enumerate(dataset, start=1):
    print(f"Example {index}")
    print(entry["question"])
    print("Reference answer:", entry["answer"])
    print("Metadata:")
    pprint(entry["metadata"], sort_dicts=False)
    print("-" * 80)

Example 1
Calculate 139 using all of these numbers: 36, 29, 95, 32, 4, 15.
Each number may be used at most once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.

Reference answer: 15 - 4 + 95 + 36 - 32 + 29
Metadata:
{'source_dataset': 'countdown',
 'source_index': 0,
 'numbers': [36, 29, 95, 32, 4, 15],
 'target': 139,
 'expression': '15 - 4 + 95 + 36 - 32 + 29',
 'difficulty': {'numbers': (4, 6), 'target': (100, 999), 'value': (1, 100)}}
--------------------------------------------------------------------------------
Example 2
Using all the numbers 74, 48, 56, 66, create an expression that equals 132.
You can only use each number once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in 

## Load a model

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from reasoning_gym.utils import SYSTEM_PROMPTS, extract_answer

model_name = "Qwen/Qwen3-4B"
device = "cuda:6"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.bfloat16, device_map=device
)
model.eval()
print(f"Loaded {model_name} on {model.device}")

/home/ujan/sac/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 348.28it/s]


Loaded Qwen/Qwen3-4B on cuda:6


## Prompt the model

Use Reasoning Gym's default system prompt: reason about the problem, then put only the final answer inside `<answer>...</answer>`.
Task-specific formatting instructions are part of the question and apply inside those tags.

In [7]:
example_index = 0
enable_thinking = True
entry = dataset[example_index]

messages = [
    {"role": "system", "content": SYSTEM_PROMPTS["default"]},
    {"role": "user", "content": entry["question"]},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
    enable_thinking=enable_thinking,
)
print(prompt)

<|im_start|>system
Given a problem, your task is to answer the question by thinking step-by-step in a clear and specific manner.
Once you have thought about the reasoning process, provide the answer in the following format:
<answer>answer here</answer>
Do not explain your reasoning inside the answer tags, provide only the final answer. When an example is provided, you should strictly follow the format of the output/answer in that example.
<|im_end|>
<|im_start|>user
Calculate 139 using all of these numbers: 36, 29, 95, 32, 4, 15.
Each number may be used at most once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.
<|im_end|>
<|im_start|>assistant



## Generate a response

In [8]:
set_seed(42)
max_new_tokens = 8192
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.6 if enable_thinking else 0.7,
        top_p=0.95 if enable_thinking else 0.8,
        top_k=20,
        min_p=0.0,
    )

output_ids = generated[0, inputs.input_ids.shape[1]:].tolist()
raw_output = tokenizer.decode(output_ids, skip_special_tokens=False)
print(raw_output)
print(f"\nGenerated tokens: {len(output_ids)}")
print("Reached token budget:", len(output_ids) == max_new_tokens)

<think>
Okay, let's see. I need to use the numbers 36, 29, 95, 32, 4, and 15 to make 139. Each number can be used only once. Hmm, how to approach this?

First, maybe I should look for combinations that can get close to 139. Let me list the numbers again: 36, 29, 95, 32, 4, 15. The target is 139. 

Looking at the numbers, 95 is the largest. If I add 95 to something, maybe? Let's see. 139 minus 95 is 44. So I need to make 44 with the remaining numbers: 36, 29, 32, 4, 15. Wait, but 36 is close to 44. 44 minus 36 is 8. Can I make 8 with the others? Let's check: 29, 32, 4, 15. Hmm, 32 minus 29 is 3, 15 minus 4 is 11. Not sure. Maybe that's not the right path.

Alternatively, maybe 32 plus something. 32 plus 36 is 68. 139 minus 68 is 71. Not sure. Maybe another approach.

What if I try multiplying some numbers? Let's see. For example, 4 times something. 4*15 is 60. 95 plus 36 is 131. 131 plus 4 is 135. Not enough. Maybe 95 plus 32 is 127. Then 127 plus 12 is 139. How to get 12? 36 divided by

## Extract and score the answer

Thinking model, so separate thinking from final response. If thinking mode never produces `</think>`, treat the final response as missing.
Then `extract_answer` returns the last complete `<answer>...</answer>` block, strips surrounding whitespace, or returns `None` when no block is found.

`dataset.score_answer` performs task-specific verification using the full entry. Some tasks compare strings; others parse numbers, expressions, or grids, and some award partial credit. Missing answers can also receive a small nonzero reward in some tasks, so inspect extraction success separately from score.

[Extraction helper and prompts](https://github.com/open-thought/reasoning-gym/blob/main/reasoning_gym/utils.py) · [Base scorer](https://github.com/open-thought/reasoning-gym/blob/main/reasoning_gym/dataset.py)

In [10]:
think_end_id = tokenizer.convert_tokens_to_ids("</think>")
if think_end_id in output_ids:
    final_start = len(output_ids) - output_ids[::-1].index(think_end_id)
else:
    final_start = len(output_ids) if enable_thinking else 0

final_response = tokenizer.decode(output_ids[final_start:], skip_special_tokens=True)
model_answer = extract_answer(final_response)
score = dataset.score_answer(answer=model_answer, entry=entry)

print("Final response:", final_response)
print("Extracted answer:", repr(model_answer))
print("Reference answer:", repr(entry["answer"]))
print("Answer extracted:", model_answer is not None)
print("Score:", score)

Final response: 

<answer>95-32+36+15+29-4</answer>
Extracted answer: '95-32+36+15+29-4'
Reference answer: '15 - 4 + 95 + 36 - 32 + 29'
Answer extracted: True
Score: 1.0
